# ♻️ Waste Classification — DCNN Model
### Multi-Class Classification: 12 Waste Categories

---
**Dataset:** Garbage Classification 12-Class (Kaggle — `mostafaabla/garbage-classification`)  
**Architecture:** Deep CNN — 6 Conv blocks + Residual Connections + BN + Dropout + Dense  
**Target Accuracy:** 93–96%  
**Framework:** TensorFlow / Keras 2.x  

---
### CNN vs DCNN Comparison
| Feature | CNN Phase 1 | DCNN Phase 2 |
|---------|-------------|-------------|
| Conv blocks | 4 | 6 |
| Max filters | 256 | 512 |
| Residual connections | No | Yes |
| Classes | 2 (binary) | 12 (multi-class) |
| Loss | Binary CE | Categorical CE |
| Output | Sigmoid (1) | Softmax (12) |
| Target accuracy | 90–93% | 93–96% |

## Cell 1 — Imports & Configuration

In [ ]:
import os, random, warnings, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, roc_auc_score, roc_curve
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# ── Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ── Hyperparameters
IMG_HEIGHT    = 224
IMG_WIDTH     = 224
BATCH_SIZE    = 32
EPOCHS        = 40
LEARNING_RATE = 5e-4   # Lower than CNN — deeper model is more sensitive
NUM_CLASSES   = 12
DROPOUT_RATE  = 0.5    # Higher than CNN for better regularization

# ── Class names (alphabetical — matches ImageDataGenerator)
CLASS_NAMES = [
    'battery','biological','brown-glass','cardboard','clothes',
    'green-glass','metal','paper','plastic','shoes','trash','white-glass'
]

CLASS_COLORS = [
    '#E24B4A','#3B6D11','#8B4513','#D4A05A','#534AB7',
    '#2E8B57','#708090','#D2691E','#185FA5','#8B008B',
    '#696969','#B0C4DE'
]

# ── Paths  (update if running locally)
# Kaggle: /kaggle/input/garbage-classification/garbage classification
DATASET_DIR = '/kaggle/input/garbage-classification/garbage classification'
# Local:  DATASET_DIR = './garbage_dataset'

print(f'TensorFlow  : {tf.__version__}')
print(f'Classes     : {NUM_CLASSES}')
print(f'Class names : {CLASS_NAMES}')

## Cell 2 — Dataset Verification & Distribution

In [ ]:
def count_per_class(directory):
    counts = {}
    for cls in sorted(os.listdir(directory)):
        p = os.path.join(directory, cls)
        if os.path.isdir(p):
            counts[cls] = len([f for f in os.listdir(p)
                                if f.lower().endswith(('.jpg','.jpeg','.png'))])
    return counts

counts = count_per_class(DATASET_DIR)
total  = sum(counts.values())

print(f'{"Class":<18} {"Count":>7}  {"Share":>6}')
print('-' * 42)
for cls, n in counts.items():
    print(f'{cls:<18} {n:>7,}  {n/total*100:>5.1f}%  {"█"*int(n/total*40)}')
print('-' * 42)
print(f'{"TOTAL":<18} {total:>7,}  100.0%')

# Bar chart
fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(list(counts.keys()), list(counts.values()),
              color=CLASS_COLORS[:len(counts)], edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, counts.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+8,
            str(val), ha='center', va='bottom', fontsize=9, fontweight='500')
ax.axhline(total/NUM_CLASSES, color='#888780', ls='--', lw=1.2, label='Mean per class')
ax.set_title('Class Distribution — 12-Class Garbage Dataset', fontsize=14, fontweight='500')
ax.set_xlabel('Waste Category', fontsize=11)
ax.set_ylabel('Image Count', fontsize=11)
ax.set_ylim(0, max(counts.values())*1.15)
ax.tick_params(axis='x', rotation=35, labelsize=10)
ax.spines[['top','right']].set_visible(False)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('dcnn_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: dcnn_class_distribution.png')

## Cell 3 — Augmentation Pipeline & Data Generators

In [ ]:
# Stronger augmentation than CNN — 12-class problem needs more variety
train_datagen = ImageDataGenerator(
    rescale            = 1./255,
    rotation_range     = 30,
    width_shift_range  = 0.20,
    height_shift_range = 0.20,
    shear_range        = 0.15,
    zoom_range         = 0.20,
    horizontal_flip    = True,
    vertical_flip      = True,          # Waste can appear upside-down
    brightness_range   = [0.80, 1.20],
    channel_shift_range= 15.0,          # Simulate different lighting colors
    fill_mode          = 'nearest',
    validation_split   = 0.20           # 80% train / 20% val
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    DATASET_DIR, target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='training', shuffle=True, seed=SEED
)
val_gen = train_datagen.flow_from_directory(
    DATASET_DIR, target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='validation', shuffle=False, seed=SEED
)
test_gen = test_datagen.flow_from_directory(
    DATASET_DIR, target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

print(f'Class mapping  : {train_gen.class_indices}')
print(f'Train samples  : {train_gen.samples:,}')
print(f'Val samples    : {val_gen.samples:,}')
print(f'Test samples   : {test_gen.samples:,}')

# Balanced class weights
cw_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_gen.classes
)
CLASS_WEIGHTS = dict(enumerate(cw_arr))
print('\nClass weights (balanced):')
for i, (cls, w) in enumerate(zip(CLASS_NAMES, cw_arr)):
    print(f'  [{i:2d}] {cls:<14}: {w:.3f}')

## Cell 4 — Sample Images Grid (All 12 Classes)

In [ ]:
n_per = 3
cls_dirs = sorted([d for d in os.listdir(DATASET_DIR)
                   if os.path.isdir(os.path.join(DATASET_DIR, d))])

fig, axes = plt.subplots(len(cls_dirs), n_per, figsize=(n_per*3.2, len(cls_dirs)*3))

for row, cls in enumerate(cls_dirs):
    cls_path = os.path.join(DATASET_DIR, cls)
    files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    sample = random.sample(files, min(n_per, len(files)))
    for col, fname in enumerate(sample):
        img = load_img(os.path.join(cls_path, fname), target_size=(IMG_HEIGHT, IMG_WIDTH))
        axes[row][col].imshow(np.array(img)/255.0)
        if col == 0:
            axes[row][col].set_ylabel(cls, fontsize=10, fontweight='500',
                                       color=CLASS_COLORS[row % len(CLASS_COLORS)])
        axes[row][col].set_xticks([]); axes[row][col].set_yticks([])

plt.suptitle('Sample Images — 12 Waste Classes', fontsize=14, fontweight='500', y=1.01)
plt.tight_layout()
plt.savefig('dcnn_class_samples.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: dcnn_class_samples.png')

## Cell 5 — DCNN Architecture

```
Input (224×224×3)
  │
  ├─ Stem Conv2D(32) → BN → ReLU
  ├─ Block 1: Conv2D(64)×2  → BN → ReLU → MaxPool → Dropout(0.25)
  ├─ Block 2: Conv2D(128)×2 → BN → ReLU → MaxPool → Dropout(0.25)
  ├─ Block 3: ResidualBlock(256) → MaxPool → Dropout(0.30)  ← skip connection
  ├─ Block 4: ResidualBlock(256) → MaxPool → Dropout(0.35)  ← skip connection
  ├─ Block 5: Conv2D(512)×2 → BN → ReLU → MaxPool → Dropout(0.40)
  ├─ Block 6: Conv2D(512)   → BN → ReLU → GlobalAvgPool → Dropout(0.45)
  ├─ Dense(1024) → BN → ReLU → Dropout(0.50)
  ├─ Dense(256)  → BN → ReLU → Dropout(0.40)
  └─ Dense(12)   → Softmax
```

In [ ]:
def residual_block(x, filters, prefix):
    """
    Residual (skip-connection) block.
    Allows gradients to bypass conv layers — prevents vanishing gradient
    in deep networks. Core idea borrowed from ResNet.

    Input  → Conv → BN → ReLU → Conv → BN
                                         + (Add)
    Input  → 1×1 Conv (project channels if needed)
    """
    shortcut = x

    # Main path
    x = layers.Conv2D(filters, (3,3), padding='same', use_bias=False, name=f'{prefix}_c1')(x)
    x = layers.BatchNormalization(name=f'{prefix}_bn1')(x)
    x = layers.Activation('relu', name=f'{prefix}_r1')(x)
    x = layers.Conv2D(filters, (3,3), padding='same', use_bias=False, name=f'{prefix}_c2')(x)
    x = layers.BatchNormalization(name=f'{prefix}_bn2')(x)

    # Projection shortcut (1×1 conv) when channel count changes
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1,1), padding='same',
                                  use_bias=False, name=f'{prefix}_proj')(shortcut)
        shortcut = layers.BatchNormalization(name=f'{prefix}_proj_bn')(shortcut)

    x = layers.Add(name=f'{prefix}_add')([x, shortcut])
    x = layers.Activation('relu', name=f'{prefix}_r2')(x)
    return x


def build_dcnn(input_shape=(224,224,3), num_classes=12, dropout=0.5):
    inputs = keras.Input(shape=input_shape, name='input')

    # Stem
    x = layers.Conv2D(32, (3,3), padding='same', use_bias=False, name='stem_conv')(inputs)
    x = layers.BatchNormalization(name='stem_bn')(x)
    x = layers.Activation('relu', name='stem_relu')(x)

    # Block 1 — 64 filters
    x = layers.Conv2D(64, (3,3), padding='same', use_bias=False, name='b1_c1')(x)
    x = layers.BatchNormalization(name='b1_bn1')(x)
    x = layers.Activation('relu', name='b1_r1')(x)
    x = layers.Conv2D(64, (3,3), padding='same', use_bias=False, name='b1_c2')(x)
    x = layers.BatchNormalization(name='b1_bn2')(x)
    x = layers.Activation('relu', name='b1_r2')(x)
    x = layers.MaxPooling2D((2,2), name='b1_pool')(x)
    x = layers.Dropout(0.25, name='b1_drop')(x)

    # Block 2 — 128 filters
    x = layers.Conv2D(128, (3,3), padding='same', use_bias=False, name='b2_c1')(x)
    x = layers.BatchNormalization(name='b2_bn1')(x)
    x = layers.Activation('relu', name='b2_r1')(x)
    x = layers.Conv2D(128, (3,3), padding='same', use_bias=False, name='b2_c2')(x)
    x = layers.BatchNormalization(name='b2_bn2')(x)
    x = layers.Activation('relu', name='b2_r2')(x)
    x = layers.MaxPooling2D((2,2), name='b2_pool')(x)
    x = layers.Dropout(0.25, name='b2_drop')(x)

    # Block 3 — 256 filters + RESIDUAL
    x = residual_block(x, 256, prefix='b3')
    x = layers.MaxPooling2D((2,2), name='b3_pool')(x)
    x = layers.Dropout(0.30, name='b3_drop')(x)

    # Block 4 — 256 filters + RESIDUAL
    x = residual_block(x, 256, prefix='b4')
    x = layers.MaxPooling2D((2,2), name='b4_pool')(x)
    x = layers.Dropout(0.35, name='b4_drop')(x)

    # Block 5 — 512 filters
    x = layers.Conv2D(512, (3,3), padding='same', use_bias=False, name='b5_c1')(x)
    x = layers.BatchNormalization(name='b5_bn1')(x)
    x = layers.Activation('relu', name='b5_r1')(x)
    x = layers.Conv2D(512, (3,3), padding='same', use_bias=False, name='b5_c2')(x)
    x = layers.BatchNormalization(name='b5_bn2')(x)
    x = layers.Activation('relu', name='b5_r2')(x)
    x = layers.MaxPooling2D((2,2), name='b5_pool')(x)
    x = layers.Dropout(0.40, name='b5_drop')(x)

    # Block 6 — 512 filters bottleneck → GAP
    x = layers.Conv2D(512, (3,3), padding='same', use_bias=False, name='b6_conv')(x)
    x = layers.BatchNormalization(name='b6_bn')(x)
    x = layers.Activation('relu', name='b6_relu')(x)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(0.45, name='b6_drop')(x)

    # Dense head
    x = layers.Dense(1024, use_bias=False, name='d1')(x)
    x = layers.BatchNormalization(name='d1_bn')(x)
    x = layers.Activation('relu', name='d1_relu')(x)
    x = layers.Dropout(dropout, name='d1_drop')(x)

    x = layers.Dense(256, use_bias=False, name='d2')(x)
    x = layers.BatchNormalization(name='d2_bn')(x)
    x = layers.Activation('relu', name='d2_relu')(x)
    x = layers.Dropout(0.40, name='d2_drop')(x)

    # Output
    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)
    return keras.Model(inputs, outputs, name='WasteDCNN')


dcnn = build_dcnn(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
                  num_classes=NUM_CLASSES, dropout=DROPOUT_RATE)
dcnn.summary()
print(f'\nTotal params     : {dcnn.count_params():,}')
print(f'Trainable params : {sum(np.prod(w.shape) for w in dcnn.trainable_weights):,}')

## Cell 6 — Compile & Callbacks

In [ ]:
dcnn.compile(
    optimizer = Adam(learning_rate=LEARNING_RATE),
    loss      = 'categorical_crossentropy',
    metrics   = [
        'accuracy',
        tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')
    ]
)

os.makedirs('dcnn_checkpoints', exist_ok=True)
os.makedirs('dcnn_logs', exist_ok=True)

cb = [
    ModelCheckpoint('dcnn_checkpoints/waste_dcnn_best.keras',
                    monitor='val_accuracy', save_best_only=True,
                    mode='max', verbose=1),
    EarlyStopping(monitor='val_loss', patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=4, min_lr=1e-7, verbose=1),
    CSVLogger('dcnn_logs/training_log.csv', append=False)
]

print('Compiled. Optimizer: Adam | Loss: categorical_crossentropy')
print('Metrics : accuracy, top3_acc')
print('Callbacks: Checkpoint | EarlyStopping(p=8) | ReduceLR | CSV')

## Cell 7 — Training

In [ ]:
print('Starting DCNN training...\n')
print(f'Train : {train_gen.samples:,}  |  Val : {val_gen.samples:,}')
print(f'Batch : {BATCH_SIZE}  |  Max epochs : {EPOCHS}')
print('-'*55)

history = dcnn.fit(
    train_gen,
    epochs          = EPOCHS,
    validation_data = val_gen,
    callbacks       = cb,
    class_weight    = CLASS_WEIGHTS,
    verbose         = 1
)

best_e   = np.argmax(history.history['val_accuracy']) + 1
best_acc = max(history.history['val_accuracy'])
best_top3= max(history.history.get('val_top3_acc', [0]))
print(f'\nBest epoch       : {best_e}')
print(f'Best val acc     : {best_acc*100:.2f}%')
print(f'Best top-3 acc   : {best_top3*100:.2f}%')

## Cell 8 — Training Curves

In [ ]:
h  = history.history
ep = range(1, len(h['accuracy'])+1)
be = np.argmax(h['val_accuracy']) + 1

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plots = [
    ('accuracy',  'val_accuracy',  'Accuracy'),
    ('loss',      'val_loss',      'Loss'),
    ('top3_acc',  'val_top3_acc',  'Top-3 Accuracy'),
]

for ax, (tk, vk, title) in zip(axes, plots):
    if tk not in h: ax.set_visible(False); continue
    ax.plot(ep, h[tk],  color='#534AB7', lw=2, marker='o', ms=3, label='Train')
    ax.plot(ep, h[vk],  color='#1D9E75', lw=2, ls='--', marker='s', ms=3, label='Val')
    ax.axvline(be, color='#E24B4A', ls=':', lw=1.5, label=f'Best ({be})')
    ax.set_title(title, fontsize=13, fontweight='500')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.legend(fontsize=10)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(True, alpha=0.3, ls='--')

bv = max(h['val_accuracy'])
axes[0].annotate(f'{bv:.4f}', xy=(be, bv), xytext=(be+0.5, bv-0.03),
                 fontsize=10, color='#E24B4A',
                 arrowprops=dict(arrowstyle='->', color='#E24B4A', lw=1.5))

plt.suptitle('DCNN Training History — 12-Class Waste Classification',
             fontsize=14, fontweight='500', y=1.02)
plt.tight_layout()
plt.savefig('dcnn_training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: dcnn_training_history.png')

## Cell 9 — Test Evaluation

In [ ]:
dcnn.load_weights('dcnn_checkpoints/waste_dcnn_best.keras')

test_res = dcnn.evaluate(test_gen, verbose=1)
print('\n' + '='*52)
for name, val in zip(dcnn.metrics_names, test_res):
    pct = f' ({val*100:.2f}%)' if 'acc' in name else ''
    print(f'  {name:<18}: {val:.4f}{pct}')

test_gen.reset()
y_pred_prob = dcnn.predict(test_gen, verbose=1)
y_pred      = np.argmax(y_pred_prob, axis=1)
y_true      = test_gen.classes

acc      = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro')
wt_f1    = f1_score(y_true, y_pred, average='weighted')
top3     = np.mean([y_true[i] in np.argsort(y_pred_prob[i])[-3:] for i in range(len(y_true))])

print(f'\n  Accuracy     : {acc:.4f} ({acc*100:.2f}%)')
print(f'  Macro F1     : {macro_f1:.4f}')
print(f'  Weighted F1  : {wt_f1:.4f}')
print(f'  Top-3 acc    : {top3:.4f} ({top3*100:.2f}%)')
print('='*52)

## Cell 10 — Confusion Matrix (12×12)

In [ ]:
cm     = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
for ax, data, fmt, title, cmap in zip(
    axes,
    [cm, cm_pct], ['d', '.0f'],
    ['Counts', 'Row %'],
    ['YlOrRd', 'Blues']
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap=cmap,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.4, linecolor='white',
                cbar_kws={'shrink': 0.75}, ax=ax)
    ax.set_title(f'Confusion Matrix — {title}', fontsize=13, fontweight='500', pad=10)
    ax.set_ylabel('True', fontsize=11)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.tick_params(axis='x', rotation=40, labelsize=9)
    ax.tick_params(axis='y', rotation=0,  labelsize=9)

print('Per-class accuracy:')
print(f'{"Class":<16}  {"Correct":>8}  {"Total":>6}  {"Acc":>7}')
print('-'*44)
for i, cls in enumerate(CLASS_NAMES):
    correct = cm[i, i]
    tot     = cm[i, :].sum()
    print(f'{cls:<16}  {correct:>8}  {tot:>6}  {correct/tot*100:>6.1f}%  {"█"*int(correct/tot*20)}')

plt.suptitle('DCNN — 12-Class Confusion Matrices', fontsize=14, fontweight='500', y=1.01)
plt.tight_layout()
plt.savefig('dcnn_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nSaved: dcnn_confusion_matrix.png')

## Cell 11 — Classification Report & Per-Class Bar Chart

In [ ]:
print('='*70)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

rep = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
f1s   = [rep[c]['f1-score']  for c in CLASS_NAMES]
precs = [rep[c]['precision'] for c in CLASS_NAMES]
recs  = [rep[c]['recall']    for c in CLASS_NAMES]

x = np.arange(len(CLASS_NAMES))
w = 0.28
fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(x-w,   precs, w, label='Precision', color='#534AB7', edgecolor='white')
ax.bar(x,     recs,  w, label='Recall',    color='#1D9E75', edgecolor='white')
ax.bar(x+w,   f1s,   w, label='F1-Score',  color='#185FA5', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=35, ha='right', fontsize=10)
ax.set_ylabel('Score', fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_title('Per-Class Precision / Recall / F1 — DCNN', fontsize=13, fontweight='500')
ax.legend(fontsize=11, loc='lower right')
ax.axhline(1.0, color='#888780', ls='--', lw=0.8, alpha=0.6)
ax.spines[['top','right']].set_visible(False)
ax.grid(True, axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig('dcnn_per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: dcnn_per_class_metrics.png')

## Cell 12 — ROC-AUC (One-vs-Rest, 12 Classes)

In [ ]:
y_bin = label_binarize(y_true, classes=np.arange(NUM_CLASSES))

fig, axes = plt.subplots(3, 4, figsize=(20, 14))
axes = axes.flatten()
auc_scores = []

for i, (cls, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_pred_prob[:, i])
    auc_val     = roc_auc_score(y_bin[:, i], y_pred_prob[:, i])
    auc_scores.append(auc_val)
    axes[i].plot(fpr, tpr, color=color, lw=2.2, label=f'AUC = {auc_val:.3f}')
    axes[i].plot([0,1],[0,1], color='#B4B2A9', lw=1.2, ls='--')
    axes[i].fill_between(fpr, tpr, alpha=0.10, color=color)
    axes[i].set_title(cls, fontsize=11, fontweight='500')
    axes[i].set_xlabel('FPR', fontsize=9)
    axes[i].set_ylabel('TPR', fontsize=9)
    axes[i].legend(fontsize=9, loc='lower right')
    axes[i].spines[['top','right']].set_visible(False)
    axes[i].grid(True, alpha=0.2)

mean_auc = np.mean(auc_scores)
print(f'Mean AUC (macro OvR): {mean_auc:.4f}')
for cls, auc in zip(CLASS_NAMES, auc_scores):
    print(f'  {cls:<16}: {auc:.4f}')

plt.suptitle(f'ROC Curves — One-vs-Rest (12 Classes)  |  Mean AUC = {mean_auc:.4f}',
             fontsize=14, fontweight='500', y=1.01)
plt.tight_layout()
plt.savefig('dcnn_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: dcnn_roc_curves.png')

## Cell 13 — Sample Predictions with Top-3 Confidence

In [ ]:
all_samples = []
for idx, cls in enumerate(sorted(os.listdir(DATASET_DIR))):
    cp = os.path.join(DATASET_DIR, cls)
    if not os.path.isdir(cp): continue
    for f in os.listdir(cp):
        if f.lower().endswith(('.jpg','.jpeg','.png')):
            all_samples.append((os.path.join(cp, f), idx))

sample = random.sample(all_samples, 16)
fig, axes = plt.subplots(4, 4, figsize=(18, 20))
axes = axes.flatten()

for i, (path, true_idx) in enumerate(sample):
    img  = load_img(path, target_size=(IMG_HEIGHT, IMG_WIDTH))
    arr  = img_to_array(img) / 255.0
    prob = dcnn.predict(np.expand_dims(arr, 0), verbose=0)[0]
    pred = int(np.argmax(prob))
    conf = prob[pred] * 100
    ok   = pred == true_idx

    top3_idx = np.argsort(prob)[::-1][:3]
    top3_str = '\n'.join([f'{CLASS_NAMES[j]}: {prob[j]*100:.1f}%' for j in top3_idx])

    axes[i].imshow(arr)
    border = '#1D9E75' if ok else '#E24B4A'
    for sp in axes[i].spines.values():
        sp.set_edgecolor(border); sp.set_linewidth(3)
    axes[i].set_title(
        f'{"✓" if ok else "✗"} Pred: {CLASS_NAMES[pred]} ({conf:.1f}%)\n'
        f'True: {CLASS_NAMES[true_idx]}\n'
        f'Top-3: {top3_str}',
        fontsize=7.5, color=border, loc='left'
    )
    axes[i].axis('off')

g = mpatches.Patch(color='#1D9E75', label='Correct')
r = mpatches.Patch(color='#E24B4A', label='Incorrect')
fig.legend(handles=[g,r], loc='lower center', ncol=2, fontsize=11,
           frameon=False, bbox_to_anchor=(0.5, -0.01))
plt.suptitle('DCNN Sample Predictions — 12-Class Waste Classification',
             fontsize=13, fontweight='500', y=1.01)
plt.tight_layout()
plt.savefig('dcnn_sample_predictions.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: dcnn_sample_predictions.png')

## Cell 14 — Grad-CAM Heatmap (Where the DCNN Looks)

In [ ]:
import cv2

def gradcam(model, img_tensor, pred_class, last_conv='b6_conv'):
    """
    Grad-CAM: compute gradient of predicted class score
    w.r.t. the last conv layer's output feature maps.
    Positive gradients (areas that increase confidence) are
    weighted and averaged to form the heatmap.
    """
    grad_model = keras.Model(
        inputs  = model.inputs,
        outputs = [model.get_layer(last_conv).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_tensor)
        loss = preds[:, pred_class]
    grads   = tape.gradient(loss, conv_out)
    weights = tf.reduce_mean(grads, axis=(0,1,2))
    cam     = conv_out[0] @ weights[..., tf.newaxis]
    cam     = tf.squeeze(cam)
    cam     = tf.maximum(cam, 0) / (tf.math.reduce_max(cam) + 1e-8)
    return cam.numpy()


# Build Grad-CAM grid (6 samples: original | heatmap | overlay)
cls_dirs = sorted([d for d in os.listdir(DATASET_DIR)
                   if os.path.isdir(os.path.join(DATASET_DIR, d))])
cam_samples = []
for idx, cls in enumerate(cls_dirs[:6]):
    cp = os.path.join(DATASET_DIR, cls)
    files = [f for f in os.listdir(cp) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    cam_samples.append((os.path.join(cp, random.choice(files)), idx))

fig, axes = plt.subplots(6, 3, figsize=(12, 21))
col_titles = ['Original', 'Grad-CAM Heatmap', 'Overlay']
for ax, t in zip(axes[0], col_titles):
    ax.set_title(t, fontsize=11, fontweight='500')

for row, (path, true_idx) in enumerate(cam_samples):
    img    = load_img(path, target_size=(IMG_HEIGHT, IMG_WIDTH))
    arr    = img_to_array(img) / 255.0
    tensor = np.expand_dims(arr, 0)
    prob   = dcnn.predict(tensor, verbose=0)[0]
    pred   = int(np.argmax(prob))
    conf   = prob[pred]*100

    try:
        hm     = gradcam(dcnn, tensor, pred)
        hm_rs  = cv2.resize(hm, (IMG_WIDTH, IMG_HEIGHT))
        hm_col = cv2.applyColorMap(np.uint8(255*hm_rs), cv2.COLORMAP_JET)
        hm_col = cv2.cvtColor(hm_col, cv2.COLOR_BGR2RGB)
        overlay= np.uint8(arr*255*0.55 + hm_col*0.45)
    except Exception as e:
        hm_rs = np.zeros((IMG_HEIGHT, IMG_WIDTH))
        hm_col = np.zeros((IMG_HEIGHT, IMG_WIDTH, 3), dtype=np.uint8)
        overlay= np.uint8(arr*255)
        print(f'Grad-CAM error: {e}')

    axes[row][0].imshow(arr)
    color = '#1D9E75' if pred==true_idx else '#E24B4A'
    axes[row][0].set_ylabel(
        f'True: {CLASS_NAMES[true_idx]}\nPred: {CLASS_NAMES[pred]} ({conf:.1f}%)',
        fontsize=8, fontweight='500', color=color
    )
    axes[row][1].imshow(hm_rs, cmap='jet')
    axes[row][2].imshow(overlay)
    for ax in axes[row]: ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Grad-CAM — Regions the DCNN Focuses On', fontsize=13, fontweight='500', y=1.01)
plt.tight_layout()
plt.savefig('dcnn_gradcam.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: dcnn_gradcam.png')

## Cell 15 — CNN vs DCNN Comparison Chart

In [ ]:
# Update CNN values from your Phase 1 notebook results
CNN_ACC  = 0.921    # Replace with actual CNN test accuracy
CNN_F1   = 0.919
CNN_AUC  = 0.965
CNN_PARAMS = 8_200_000

metrics   = ['Accuracy', 'F1 Score', 'AUC-ROC']
cnn_vals  = [CNN_ACC, CNN_F1, CNN_AUC]
dcnn_vals = [acc,     macro_f1, mean_auc]
x = np.arange(len(metrics))
w = 0.32

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Metric bars
ax = axes[0]
b1 = ax.bar(x-w/2, cnn_vals,  w, label='CNN  (2 classes)',  color='#378ADD', edgecolor='white')
b2 = ax.bar(x+w/2, dcnn_vals, w, label='DCNN (12 classes)', color='#534AB7', edgecolor='white')
for bar, val in zip(list(b1)+list(b2), cnn_vals+dcnn_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.004,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='500')
ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_ylim(0.80, 1.06)
ax.set_title('CNN vs DCNN Performance', fontsize=13, fontweight='500')
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
ax.grid(True, axis='y', alpha=0.25)

# Architecture table
ax = axes[1]
ax.axis('off')
tdata = [
    ['Feature',          'CNN',                      'DCNN'],
    ['Conv blocks',      '4',                        '6'],
    ['Max filters',      '256',                      '512'],
    ['Residual blocks',  'No',                       'Yes (Blocks 3 & 4)'],
    ['Classes',          '2',                        '12'],
    ['Output layer',     'Sigmoid (1)',               'Softmax (12)'],
    ['Loss function',    'Binary CE',                'Categorical CE'],
    ['Parameters',       f'{CNN_PARAMS/1e6:.1f}M',   f'{dcnn.count_params()/1e6:.1f}M'],
    ['Accuracy',         f'{CNN_ACC*100:.1f}%',       f'{acc*100:.1f}%'],
    ['Macro F1',         f'{CNN_F1:.3f}',             f'{macro_f1:.3f}'],
    ['Mean AUC',         f'{CNN_AUC:.3f}',            f'{mean_auc:.3f}'],
]
tbl = ax.table(cellText=tdata[1:], colLabels=tdata[0], loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.1, 1.65)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#534AB7')
        cell.set_text_props(color='white', fontweight='bold')
    elif col == 1: cell.set_facecolor('#EBF3FB')
    elif col == 2: cell.set_facecolor('#F0EEFB')
    cell.set_edgecolor('#D3D1C7')
ax.set_title('Architecture Comparison', fontsize=13, fontweight='500', pad=16)

plt.suptitle('Phase 1 CNN vs Phase 2 DCNN — Complete Comparison',
             fontsize=14, fontweight='500', y=1.02)
plt.tight_layout()
plt.savefig('cnn_vs_dcnn_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cnn_vs_dcnn_comparison.png')

## Cell 16 — Final Report Card

In [ ]:
n_epochs = len(history.history['accuracy'])
print('\n' + '='*60)
print('    DCNN WASTE CLASSIFIER — FINAL REPORT CARD')
print('='*60)
rows = [
    ['Model',            'WasteDCNN (6-Block + Residual Connections)'],
    ['Dataset',          'Garbage Classification 12-Class (Kaggle)'],
    ['Task',             '12-Class: battery→white-glass'],
    ['Input size',       f'{IMG_HEIGHT}×{IMG_WIDTH}×3'],
    ['Parameters',       f'{dcnn.count_params():,}'],
    ['Train samples',    f'{train_gen.samples:,}'],
    ['Val samples',      f'{val_gen.samples:,}'],
    ['Epochs run',       f'{n_epochs} / {EPOCHS} (EarlyStopping)'],
    ['───────────────',  '───────────────────────────────────────'],
    ['Test Accuracy',    f'{acc:.4f}  ({acc*100:.2f}%)'],
    ['Top-3 Accuracy',   f'{top3:.4f}  ({top3*100:.2f}%)'],
    ['Macro F1',         f'{macro_f1:.4f}'],
    ['Weighted F1',      f'{wt_f1:.4f}'],
    ['Mean AUC (OvR)',   f'{mean_auc:.4f}'],
    ['───────────────',  '───────────────────────────────────────'],
    ['Best checkpoint',  'dcnn_checkpoints/waste_dcnn_best.keras'],
    ['Training log',     'dcnn_logs/training_log.csv'],
]
for row in rows:
    print(f'  {row[0]:<20}: {row[1]}')
print('='*60)

delta = (acc - CNN_ACC) * 100
print(f'\n  Improvement over CNN : {delta:+.2f}% accuracy points')

## Cell 17 — Save Model & Export

In [ ]:
os.makedirs('saved_models_dcnn', exist_ok=True)

# Full Keras model
dcnn.save('saved_models_dcnn/waste_dcnn_final.keras')
print('Saved: saved_models_dcnn/waste_dcnn_final.keras')

# TF SavedModel (TF Serving / FastAPI)
dcnn.export('saved_models_dcnn/waste_dcnn_savedmodel')
print('Saved: saved_models_dcnn/waste_dcnn_savedmodel/')

# Weights only
dcnn.save_weights('saved_models_dcnn/waste_dcnn_weights.h5')
print('Saved: saved_models_dcnn/waste_dcnn_weights.h5')

# Config JSON for inference script
config = {
    'model_name'   : 'WasteDCNN',
    'model_path'   : 'saved_models_dcnn/waste_dcnn_final.keras',
    'class_names'  : CLASS_NAMES,
    'class_indices': {v: k for k, v in train_gen.class_indices.items()},
    'num_classes'  : NUM_CLASSES,
    'img_height'   : IMG_HEIGHT,
    'img_width'    : IMG_WIDTH,
    'test_accuracy': round(float(acc), 4),
    'macro_f1'     : round(float(macro_f1), 4),
    'mean_auc'     : round(float(mean_auc), 4),
}
with open('saved_models_dcnn/dcnn_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Saved: saved_models_dcnn/dcnn_config.json')
print('\nAll exports complete — ready for FastAPI + React frontend.')

## Cell 18 — Inference Helper (FastAPI Ready)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Paste this into your FastAPI /predict endpoint
# ─────────────────────────────────────────────────────────────────────────────
from tensorflow.keras.models import load_model as _load

def load_dcnn(model_path='saved_models_dcnn/waste_dcnn_final.keras',
              config_path='saved_models_dcnn/dcnn_config.json'):
    model = _load(model_path, compile=False)
    with open(config_path) as f:
        cfg = json.load(f)
    return model, cfg


def predict_waste(image_path, model, config, top_k=3):
    """
    Returns:
        {
          'predicted_class': 'plastic',
          'confidence': 87.3,
          'top_k': [
            {'class': 'plastic',  'probability': 87.3},
            {'class': 'metal',    'probability':  7.1},
            {'class': 'cardboard','probability':  3.2}
          ]
        }
    """
    img  = load_img(image_path, target_size=(config['img_height'], config['img_width']))
    arr  = img_to_array(img) / 255.0
    prob = model.predict(np.expand_dims(arr, 0), verbose=0)[0]
    topk = np.argsort(prob)[::-1][:top_k]
    cls  = config['class_names']
    return {
        'predicted_class': cls[int(np.argmax(prob))],
        'confidence'     : round(float(np.max(prob))*100, 2),
        'top_k'          : [{'class': cls[int(i)],
                              'probability': round(float(prob[i])*100, 2)}
                             for i in topk]
    }


# Demo
m, cfg = load_dcnn()
demo   = os.path.join(DATASET_DIR, 'plastic',
                       os.listdir(os.path.join(DATASET_DIR,'plastic'))[0])
result = predict_waste(demo, m, cfg)

print('DCNN INFERENCE DEMO')
print('='*45)
print(f'Prediction  : {result["predicted_class"]}')
print(f'Confidence  : {result["confidence"]}%')
print('Top-3:')
for r in result['top_k']:
    print(f'  {r["class"]:<16}: {r["probability"]}%')
print('='*45)

---
## ✅ DCNN Notebook Complete

| Item | Detail |
|------|--------|
| Architecture | 6-block DCNN with residual connections |
| Dataset | 12-class garbage (15,515 images) |
| Expected accuracy | **93–96%** |
| Saved model | `saved_models_dcnn/waste_dcnn_final.keras` |
| TF Serving | `saved_models_dcnn/waste_dcnn_savedmodel/` |

### Output Files
| File | Description |
|------|-------------|
| `dcnn_class_distribution.png` | 12-class bar chart |
| `dcnn_class_samples.png` | image grid per class |
| `dcnn_training_history.png` | accuracy / loss / top-3 curves |
| `dcnn_confusion_matrix.png` | 12×12 confusion matrix |
| `dcnn_per_class_metrics.png` | precision / recall / F1 bars |
| `dcnn_roc_curves.png` | 12 OvR ROC curves |
| `dcnn_sample_predictions.png` | predictions + top-3 confidence |
| `dcnn_gradcam.png` | Grad-CAM heatmaps |
| `cnn_vs_dcnn_comparison.png` | side-by-side comparison |

### Next: Frontend Integration
Use `predict_waste()` in Cell 18 as the core of a **FastAPI `/predict` endpoint**.  
Your **React frontend** uploads an image → calls the endpoint → displays class + confidence bar.